# Notable Encounters

Deep-dive into the most scientifically interesting encounters in the catalog:
- Closest approach events
- Encounters involving the largest asteroids
- NEA–MBA encounters
- Encounters near Gaia's observability limit

In [ ]:
from pathlib import Path
import polars as pl
import plotly.express as px

from src.catalog.query import load_catalog, filter_encounters, top_encounters

df = load_catalog(Path('../data/output/encounters_characterized.parquet'))
print(f'{len(df):,} encounters loaded')

## Top 20 closest approaches

In [ ]:
closest = top_encounters(df, n=20, by='dist_au', ascending=True)
closest.select([
    'number_1', 'number_2', 'designation_1', 'designation_2',
    'date_utc', 'dist_au', 'dist_km', 'rel_vel_km_s',
    'diameter_1_km', 'diameter_2_km', 'gaia_observable'
])

## Encounters involving major bodies (Ceres, Vesta, Hygiea)

In [ ]:
major_bodies = {1: 'Ceres', 4: 'Vesta', 10: 'Hygiea', 65: 'Cybele', 88: 'Thisbe', 511: 'Davida'}
for num, name in major_bodies.items():
    hits = filter_encounters(df, body_ids=[num])
    if len(hits) == 0:
        print(f'({num}) {name}: no encounters < 0.01 AU')
    else:
        print(f'({num}) {name}: {len(hits)} encounters')
        display(hits.sort('dist_au').select([
            'number_1', 'number_2', 'date_utc', 'dist_au', 'rel_vel_km_s', 'gaia_observable'
        ]))

## NEA encounters

In [ ]:
nea_enc = df.filter(
    (pl.col('class_1') == 'NEA') | (pl.col('class_2') == 'NEA')
).sort('dist_au')

print(f'NEA encounters: {len(nea_enc)}')
nea_enc.head(10).select([
    'number_1', 'number_2', 'date_utc', 'dist_au',
    'rel_vel_km_s', 'class_1', 'class_2', 'gaia_observable'
])

## Encounters with large asteroids (diameter > 100 km)

In [ ]:
big = df.filter(
    (pl.col('diameter_1_km') > 100) | (pl.col('diameter_2_km') > 100)
).sort('dist_au')

print(f'Encounters with D > 100 km: {len(big)}')
big.head(15).select([
    'number_1', 'number_2', 'date_utc', 'dist_au', 'rel_vel_km_s',
    'diameter_1_km', 'diameter_2_km', 'gaia_observable'
])

## Fastest encounters

In [ ]:
fastest = top_encounters(df, n=10, by='rel_vel_km_s', ascending=False)
fastest.select([
    'number_1', 'number_2', 'date_utc', 'dist_au',
    'rel_vel_km_s', 'class_1', 'class_2'
])

## Gaia-observable encounters with large bodies

In [ ]:
prime = filter_encounters(
    df,
    max_dist_au=0.005,
    gaia_observable_only=True,
).sort('dist_au')

print(f'Gaia-observable encounters < 0.005 AU: {len(prime)}')
prime.head(20).select([
    'number_1', 'number_2', 'date_utc', 'dist_au', 'rel_vel_km_s',
    'diameter_1_km', 'diameter_2_km', 'solar_elongation_deg'
])

## Size vs velocity scatter for top encounters

In [ ]:
top500 = top_encounters(df, n=500, by='dist_au').drop_nulls(['diameter_1_km'])

fig = px.scatter(
    top500.to_pandas(),
    x='dist_au',
    y='rel_vel_km_s',
    size='diameter_1_km',
    color='gaia_observable',
    hover_data=['number_1', 'number_2', 'date_utc', 'diameter_1_km'],
    color_discrete_map={True: '#2ecc71', False: '#e74c3c'},
    size_max=30,
    title='Top 500 closest encounters: distance vs velocity (bubble size = diameter body 1)',
    labels={
        'dist_au': 'Min. separation (AU)',
        'rel_vel_km_s': 'Relative velocity (km/s)',
    },
)
fig.show()